# Field Shift · Punto 4: años que rindieron distinto de lo esperado

**Qué ve el productor:** en la historia de su condado, los años que se salieron de lo normal, con el motivo más probable. Por ejemplo:

> "In 2012, corn in this county yielded about {X} % less than in a normal year. July heat tends to explain most of it."

**Cómo se hace**, con el modelo del punto 2 (no se entrena nada nuevo):

1. **Rinde de un año normal:** el modelo predice el rinde de cada año usando el **clima típico** del condado en vez del clima de ese año. Así queda lo que se esperaba con ese suelo, esa rotación y esa tecnología.
2. **Diferencia:** rinde real − rinde de un año normal. Si se aparta más que `UMBRAL_PCT`, el año se marca.
3. **Por qué:** cuánto aportó cada grupo de clima respecto de un año normal (calor de julio, primavera húmeda, etc.), con los valores SHAP del modelo. El grupo que más pesó es la causa principal.

**Antes de correrla:** correr `punto2_rotaciones.ipynb` con maíz y con soja (crea los modelos en `backend/ML/modelos/` y `backend/Datos/tabla_modelado.csv`). Si faltan, cada paso dice "en espera".

**Cómo usarla:** elegir `CULTIVO` en el paso 1, correr todo y repetir con el otro cultivo. Las salidas de los dos se juntan en el mismo archivo.

## 0. Librerías

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

## 1. Parámetros

- `UMBRAL_PCT`: cuánto tiene que apartarse el rinde real del de un año normal para marcar el año. Con 10 %, en maíz es unos 18 bu/ac, parecido al error promedio del modelo del punto 2: menos que eso no se distingue del ruido.
- `GRUPOS_CLIMA`: los mismos grupos y nombres que propone el punto 3 (`PUNTO2.md`, sección "Para el punto 3"). Si cambian allá, cambialos acá.
- Solo se usan los años de desarrollo (2009–2023). 2024–2025 son la prueba final del punto 2 y no se miran todavía.

In [ ]:
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
SALIDA = RAIZ / "backend" / "Datos"

CULTIVO = "maiz"            # "maiz" o "soja": se corre una vez por cultivo
UMBRAL_PCT = 10             # % de diferencia para marcar un año
ANIOS_PRUEBA = [2024, 2025]
NOMBRES_ESTADO = {"17": "Illinois", "18": "Indiana", "19": "Iowa"}

# Solo grupos de clima: suelo y rotación no cambian de un año a otro dentro del condado
GRUPOS_CLIMA = {
    "calor_julio": ["tmax_jul_c", "dias_tmax_gt30_jul", "dias_tmax_gt35_07"],
    "agua_julio": ["prec_jul_mm", "deficit_mm_07"],
    "calor_agosto": ["tmax_c_08"],
    "agua_agosto": ["prec_ago_mm", "deficit_mm_08"],
    "temporada": ["prec_temporada_mm", "gdd_temporada", "deficit_temporada_mm"],
    "primavera_humeda": ["prec_power_mm_04", "prec_power_mm_05", "dias_lluvia_gt50_04", "dias_lluvia_gt50_05",
                         "gwettop_04", "gwettop_05", "gwetroot_04", "gwetroot_05"],
}

ruta_modelo = RAIZ / "backend" / "ML" / "modelos" / f"punto2_rotaciones_{CULTIVO}.joblib"
ruta_tabla = SALIDA / "tabla_modelado.csv"
HAY_MODELO = ruta_modelo.exists() and ruta_tabla.exists()
print("Cultivo:", CULTIVO, "| ¿Está el modelo del punto 2?", HAY_MODELO)
if not HAY_MODELO:
    print("Falta correr punto2_rotaciones.ipynb con este cultivo. La notebook queda en espera.")

## 2. Cargar el modelo y la tabla del punto 2

El modelo predice la **anomalía** (rinde − tendencia del estado). Para volver a bu/ac se le suma la tendencia, que viene guardada junto con el modelo.

In [ ]:
if HAY_MODELO:
    paquete = joblib.load(ruta_modelo)
    tabla = pd.read_csv(ruta_tabla, dtype={"fips": str, "estado": str})
    tabla = tabla[(tabla["cultivo"] == CULTIVO) & ~tabla["anio"].isin(ANIOS_PRUEBA)].reset_index(drop=True)
    COLUMNAS_CLIMA = [c for cols in GRUPOS_CLIMA.values() for c in cols if c in paquete["features"]]
    print(f"Modelo: {type(paquete['modelo'].named_steps['modelo']).__name__} | datos {paquete['version_datos']} "
          f"| {len(tabla)} filas condado-año | {len(COLUMNAS_CLIMA)} columnas de clima")
else:
    print("En espera del modelo del punto 2.")

## 3. Funciones del punto 2

Copias de `separar_rotacion` y de la tendencia del punto 2, para usar el modelo guardado igual que allá. Si cambian en el punto 2, cambialas acá.

In [ ]:
def separar_rotacion(datos, medias):
    d = datos.copy()
    for c in paquete["rotacion"]:
        media = d["fips"].map(medias[c]).fillna(d.groupby("fips")[c].transform("mean"))
        d[f"{c}_media"] = media
        d[f"{c}_desvio"] = d[c] - media
    return d


def aplicar_tendencia(coefs, datos):
    return np.array([np.polyval(coefs["por_estado"].get(e, coefs["global"]), a)
                     for e, a in zip(datos["estado"], datos["anio"])])


def predecir_con_contribuciones(datos):
    """Rinde predicho (bu/ac) y la contribución SHAP de cada columna (bu/ac)."""
    x = separar_rotacion(datos, paquete["medias"])[paquete["features"]]
    prep = paquete["modelo"].named_steps["prep"]
    arboles = paquete["modelo"].named_steps["modelo"]
    contrib = arboles.predict(prep.transform(x), pred_contrib=True)
    nombres = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]
    shap = pd.DataFrame(contrib[:, :-1], columns=nombres, index=datos.index)
    return contrib.sum(axis=1) + aplicar_tendencia(paquete["tendencia"], datos), shap

## 4. Rinde de un año normal

Para cada condado y año se arma una copia con el **clima típico del condado** (la mediana de sus años 2009–2023) y todo lo demás igual: el mismo suelo, la misma rotación y la misma tendencia de ese año. Lo que predice el modelo con esa copia es el rinde de un año normal.

Se usa la mediana y no el promedio para que un año extremo como 2012 no corra lo que se considera normal.

In [ ]:
if HAY_MODELO:
    normal = tabla.copy()
    clima_tipico = tabla.groupby("fips")[COLUMNAS_CLIMA].median()
    normal[COLUMNAS_CLIMA] = clima_tipico.loc[tabla["fips"]].to_numpy()

    pred_real, shap_real = predecir_con_contribuciones(tabla)
    pred_normal, shap_normal = predecir_con_contribuciones(normal)

    res = tabla[["fips", "anio", "estado", "cultivo", "rend_bu_ac"]].copy()
    res["rend_anio_normal_bu_ac"] = pred_normal
    res["rend_predicho_bu_ac"] = pred_real
    res["diferencia_bu_ac"] = res["rend_bu_ac"] - res["rend_anio_normal_bu_ac"]
    res["diferencia_pct"] = res["diferencia_bu_ac"] / res["rend_anio_normal_bu_ac"] * 100
    display(res.describe()[["rend_bu_ac", "rend_anio_normal_bu_ac", "diferencia_pct"]].round(1))
else:
    print("En espera del modelo del punto 2.")

## 5. Por qué: el clima de ese año contra el de un año normal

Para cada grupo de clima se resta su contribución en el año real menos su contribución en el año normal. Eso dice **cuántos bu/ac le sumó o le restó ese grupo** respecto de un año normal.

Se separan dos partes de la diferencia:

- **explicada por el clima:** la suma de los grupos (lo que el modelo atribuye al clima del año);
- **no explicada:** rinde real − rinde predicho con el clima real. Es lo que el modelo no ve: granizo, inundaciones, plagas, errores del dato o simplemente error del modelo.

La **causa principal** es el grupo con más peso, siempre que vaya en la misma dirección que la diferencia. Si lo no explicado pesa más que todo el clima junto, la causa queda como `no_explicado`.

In [ ]:
if HAY_MODELO:
    efecto = pd.DataFrame({g: (shap_real[[c for c in cols if c in shap_real]].sum(axis=1)
                               - shap_normal[[c for c in cols if c in shap_normal]].sum(axis=1))
                           for g, cols in GRUPOS_CLIMA.items()})
    res["explicado_clima_bu_ac"] = efecto.sum(axis=1)
    res["no_explicado_bu_ac"] = res["rend_bu_ac"] - res["rend_predicho_bu_ac"]

    signo = np.sign(res["diferencia_bu_ac"])
    a_favor = efecto.mul(signo, axis=0)                     # positivo = empuja en la dirección de la diferencia
    res["causa_principal"] = a_favor.idxmax(axis=1)
    columna_causa = efecto.columns.get_indexer(res["causa_principal"])
    res["efecto_causa_bu_ac"] = efecto.to_numpy()[np.arange(len(efecto)), columna_causa]
    sin_causa = (a_favor.max(axis=1) <= 0) | (res["no_explicado_bu_ac"].abs() > res["explicado_clima_bu_ac"].abs())
    res.loc[sin_causa, "causa_principal"] = "no_explicado"
    res.loc[sin_causa, "efecto_causa_bu_ac"] = res.loc[sin_causa, "no_explicado_bu_ac"]

    res["marcado"] = res["diferencia_pct"].abs() >= UMBRAL_PCT
    res["direccion"] = np.where(res["diferencia_bu_ac"] < 0, "peor", "mejor")
    print(f"Años marcados: {res['marcado'].mean() * 100:.0f} % de los condado-año (umbral {UMBRAL_PCT} %)")
    display(pd.crosstab(res.loc[res["marcado"], "causa_principal"], res.loc[res["marcado"], "direccion"], margins=True))
else:
    print("En espera del modelo del punto 2.")

## 6. Chequeo: ¿reconoce los años que todos conocen?

Si el método funciona, tiene que encontrar los años famosos del Corn Belt:

- **2012:** sequía y calor en julio. La mayoría de los condados tienen que salir marcados como `peor`, con causa de calor o falta de agua.
- **2019:** primavera muy húmeda, siembra atrasada. Tendría que aparecer `primavera_humeda` como causa en bastantes condados.

Por año: qué % de condados quedó marcado, la diferencia mediana y la causa más común entre los marcados.

In [ ]:
if HAY_MODELO:
    marcados = res[res["marcado"]]
    por_anio = pd.DataFrame({
        "% condados marcados": res.groupby("anio")["marcado"].mean() * 100,
        "diferencia mediana %": res.groupby("anio")["diferencia_pct"].median(),
        "causa más común (marcados)": marcados.groupby("anio")["causa_principal"].agg(lambda s: s.mode().iat[0]),
    })
    display(por_anio.round(0))

    r2012 = res[res["anio"] == 2012]
    ok_2012 = (r2012["marcado"] & (r2012["direccion"] == "peor")).mean()
    calor_agua = r2012.loc[r2012["marcado"], "causa_principal"].isin(["calor_julio", "agua_julio", "temporada",
                                                                      "calor_agosto", "agua_agosto"]).mean()
    print(f"2012: {ok_2012 * 100:.0f} % de los condados marcados como peor; "
          f"en {calor_agua * 100:.0f} % de esos la causa es calor o falta de agua.")
    if ok_2012 < 0.5:
        print("ATENCIÓN: no reconoce la sequía de 2012. Revisar antes de mostrarlo.")
else:
    print("En espera del modelo del punto 2.")

## 7. Ejemplo: la historia de un condado

Lo que vería el productor en la línea de tiempo: cada año con su diferencia y, si está marcado, la causa.

In [ ]:
FIPS_EJEMPLO = "19169"   # Story, Iowa

if HAY_MODELO:
    ej = res[res["fips"] == FIPS_EJEMPLO].set_index("anio")
    display(ej[["rend_bu_ac", "rend_anio_normal_bu_ac", "diferencia_pct", "marcado", "causa_principal",
                "efecto_causa_bu_ac"]].round(1))
else:
    print("En espera del modelo del punto 2.")

## 8. Guardar

`backend/Datos/anomalias_anio.csv`, una fila por condado, año y cultivo:

| Columna | Qué es |
|---|---|
| `fips, anio, estado, cultivo` | Claves |
| `rend_bu_ac` | Rinde real (NASS) |
| `rend_anio_normal_bu_ac` | Rinde predicho con el clima típico del condado |
| `rend_predicho_bu_ac` | Rinde predicho con el clima real de ese año |
| `diferencia_bu_ac`, `diferencia_pct` | Real − año normal |
| `marcado` | `True` si `diferencia_pct` se aparta `UMBRAL_PCT` o más |
| `direccion` | `peor` o `mejor` que un año normal |
| `causa_principal` | Id del grupo de clima que más pesó, o `no_explicado` |
| `efecto_causa_bu_ac` | Cuántos bu/ac aportó esa causa respecto de un año normal |
| `explicado_clima_bu_ac` | Suma de todos los grupos de clima |
| `no_explicado_bu_ac` | Real − predicho con el clima real |
| `ef_<grupo>_bu_ac` | Aporte de cada grupo de clima, para quien quiera más detalle |

Los nombres para la app de cada `causa_principal` van en `textos.json` (punto 3).

In [ ]:
def guardar_por_cultivo(df, ruta):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    if ruta.exists():
        viejo = pd.read_csv(ruta, dtype={"fips": str, "estado": str})
        df = pd.concat([viejo[viejo["cultivo"] != CULTIVO], df], ignore_index=True)
    df.to_csv(ruta, index=False)


if HAY_MODELO:
    salida = pd.concat([res, efecto.add_prefix("ef_").add_suffix("_bu_ac")], axis=1)
    salida = salida.round({c: 1 for c in salida.columns if salida[c].dtype.kind == "f"})
    guardar_por_cultivo(salida, SALIDA / "anomalias_anio.csv")
    print(f"Guardadas {len(salida)} filas en backend/Datos/anomalias_anio.csv")
else:
    print("En espera del modelo del punto 2.")

## Próximos pasos

- [ ] Acordar con el front dónde va la línea de tiempo del condado (`INFO.md` no tiene un archivo para esto todavía) y generarlo en `backend/Datos/front/`.
- [ ] Frases para cada `causa_principal` en `textos.json`, junto con el punto 3. Por ejemplo: *"In {anio}, corn yielded about {pct} % {less/more} than in a normal year. {July heat} tends to explain most of it."*
- [ ] Hacer las predicciones fuera del fold (reentrenando sin cada año), para que el año explicado no esté en el entrenamiento.
- [ ] Cuando se mire la prueba del punto 2, sumar 2024 y 2025.
- [ ] Clasificar cada año como seco, normal o húmedo con la misma regla que el punto 1 (`clase_anio`), cuando esté definida.